<a href="https://colab.research.google.com/github/tanveer-builds/finetunning-llm/blob/main/CPT_Math_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Continued Pretraining (CPT) — Full Finetuning on Math Domain

**Model:** `Qwen/Qwen3-0.6B-Base`  
**Dataset:** `pritamdeb68/Math-Pretraining-Data`  
**Technique:** Full Finetuning (no LoRA) + Sequence Packing  

---

### ⚠️ Before You Start
1. Go to **Runtime → Change runtime type → T4 GPU** (free) or **A100** (Colab Pro)
2. Add your secrets in the 🔑 **Colab Secrets panel** (left sidebar):
   - `HF_TOKEN` — your Hugging Face write token
   - `COMET_API_KEY` — your Comet ML API key
   - `COMET_PROJECT_NAME` — your Comet project name (e.g. `finetuning-sessions-week1`)

## 📦 Cell 1 — Install Dependencies

> This takes 3–5 minutes on first run. Unsloth compiles custom CUDA kernels.

In [ ]:
import os
os.environ['UNSLOTH_FORCE_FLOAT32'] = '1'
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'
print("✅ Env vars set")

✅ Env vars set


In [ ]:
# Install all required packages
# Unsloth must be installed BEFORE transformers/trl to avoid version conflicts
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "trl>=0.15.0" "transformers>=4.51.0" "datasets>=3.0.0"
!pip install --quiet "comet_ml>=3.44.0" "huggingface_hub>=0.23.0" fire

print("✅ All packages installed successfully!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.8/785.8 kB 17.6 MB/s 

In [ ]:
!pip install modelscope

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 48.0 MB/s eta 0:00:00


## 🔐 Cell 2 — Load Secrets & Authenticate

> Uses Colab's built-in Secrets manager — your keys are never exposed in the notebook.

In [ ]:
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

In [ ]:
import os
from google.colab import userdata

# --- Load from Colab Secrets panel (🔑 left sidebar) ---
os.environ["HF_TOKEN"]           = userdata.get("HF_TOKEN")
os.environ["COMET_API_KEY"]      = userdata.get("COMET_API_KEY")
os.environ["COMET_PROJECT_NAME"] = userdata.get("COMET_PROJECT_NAME")

HF_TOKEN      = os.environ["HF_TOKEN"]
COMET_API_KEY = os.environ["COMET_API_KEY"]
COMET_PROJECT = os.environ["COMET_PROJECT_NAME"]

# --- Authenticate with Hugging Face Hub ---
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to Hugging Face Hub.")

## 🖥️ Cell 3 — Verify GPU

> Flash Attention 2 (used by Unsloth) requires Compute Capability ≥ 8.0 (Ampere+).  
> On free Colab T4 (CC 7.5), Unsloth falls back gracefully. Use A100/L4 for best speed.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected. Go to Runtime → Change runtime type → GPU.")

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024**3
cc = f"{gpu.major}.{gpu.minor}"

print(f"✅ GPU Detected!")
print(f"   Name             : {gpu.name}")
print(f"   Compute Capability: {cc}  {'⚡ Flash Attention 2 supported!' if gpu.major >= 8 else '⚠️  CC < 8.0 — Unsloth will use fallback (slower)'}")
print(f"   Total VRAM       : {vram_gb:.2f} GB")
print()

# Friendly VRAM guidance
if vram_gb < 12:
    print("💡 Tip: <12 GB VRAM detected. Reduce max_seq_length to 512 or batch_size to 2 if you hit OOM.")
elif vram_gb < 20:
    print("💡 Tip: 12–20 GB VRAM. Default settings should work. Reduce batch_size if OOM.")
else:
    print("💡 Tip: >20 GB VRAM — you can increase batch_size or max_seq_length for faster training!")

## ⚙️ Cell 4 — Configuration

> **Edit this cell** to customise your run. All other cells read from these variables.

In [ ]:
# ============================================================
#   ✏️  EDIT THESE VALUES TO CUSTOMISE YOUR TRAINING RUN
# ============================================================

# --- Model ---
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
# Other options: "Qwen/Qwen2.5-0.5B", "HuggingFaceTB/SmolLM2-360M"

# --- Dataset ---
DATASET_NAME    = "pritamdeb68/Math-Pretraining-Data"
DATASET_SAMPLES = 10000   # Increase for a real production run (use None for full dataset)
TEXT_COLUMN     = "text"  # Column name that contains raw text in the dataset

# --- Output ---
OUTPUT_DIR   = "/content/outputs"   # Local checkpoint directory
HUB_MODEL_ID = "Qwen3-0.6B-Base-CPT-Math"  # Your HF repo name (will be: username/this-name)

# --- Training Hyperparameters ---
MAX_SEQ_LENGTH              = 1024  # Reduce to 512 if you hit OOM on T4
BATCH_SIZE                  = 1     # Reduce to 2 if you hit OOM
GRADIENT_ACCUMULATION_STEPS = 4     # Effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LEARNING_RATE               = 2e-5  # Low LR for full finetune (prevents catastrophic forgetting)
NUM_TRAIN_EPOCHS            = 1
WARMUP_STEPS                = 100
WEIGHT_DECAY                = 0.01
LOGGING_STEPS               = 20
SEED                        = 3407

# ============================================================
print("✅ Configuration set:")
print(f"   Model      : {MODEL_NAME}")
print(f"   Dataset    : {DATASET_NAME} ({DATASET_SAMPLES} samples)")
print(f"   Seq Length : {MAX_SEQ_LENGTH}")
print(f"   Batch Size : {BATCH_SIZE} × {GRADIENT_ACCUMULATION_STEPS} accum = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} effective")
print(f"   Hub Target : {HUB_MODEL_ID}")

## 🤖 Cell 5 — Load Model

> Full Finetuning loads raw weights in bfloat16 (no 4-bit quantization).  
> This gives us real gradients for all ~600M parameters.

In [ ]:
import logging as log
import sys

# --- Setup Logging ---
log.basicConfig(
    level=log.INFO,
    format='%(asctime)s — %(levelname)s — %(message)s',
    handlers=[log.StreamHandler(sys.stdout)]
)
logger = log.getLogger(__name__)

from unsloth import FastLanguageModel

logger.info(f"Loading base model: {MODEL_NAME} ...")
logger.info("Note: load_in_4bit=False → full precision weights (needed for Full Finetuning)")

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = torch.float32,
    load_in_4bit = False,        # ← CRITICAL: Full Finetuning requires full precision
    token        = HF_TOKEN,
)
logger.info("✅ Model loaded successfully in full precision.")

## 📚 Cell 6 — Load Dataset

> Continued Pretraining uses **raw text** (no Q&A pairs). We inject math documents.

In [ ]:
from datasets import load_dataset

logger.info(f"Loading dataset: {DATASET_NAME} ...")

split_str = f"train[:{DATASET_SAMPLES}]" if DATASET_SAMPLES else "train"
dataset = load_dataset(DATASET_NAME, split=split_str)

logger.info(f"✅ Dataset loaded. Samples: {len(dataset)}")

# --- Inspect the data ---
print(f"\n📋 Dataset Info:")
print(f"   Columns : {dataset.column_names}")
print(f"   Samples : {len(dataset)}")
print(f"\n📄 Sample entry (first 300 chars):")
print("-" * 60)

# Show first entry, handling different column name possibilities
sample = dataset[0]
if TEXT_COLUMN in sample:
    print(str(sample[TEXT_COLUMN])[:300] + "...")
else:
    print(f"⚠️  Column '{TEXT_COLUMN}' not found. Available: {list(sample.keys())}")
    print("Update TEXT_COLUMN in Cell 4 to match your dataset.")
print("-" * 60)

## 🏋️ Cell 7 — Configure Trainer

> `packing=True` concatenates short docs into full-length blocks — this is the  
> key technique for efficient Continued Pretraining.

In [ ]:
import shutil, os
cache_path = "/content/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("✅ Unsloth cache cleared")

# Force accelerate to use NO mixed precision
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

In [ ]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    warmup_steps                = WARMUP_STEPS,
    learning_rate               = LEARNING_RATE,
    num_train_epochs            = NUM_TRAIN_EPOCHS,
    weight_decay                = WEIGHT_DECAY,
    seed                        = SEED,
    bf16                        = False,
    fp16                        = False,
    optim                       = "adamw_torch",
    logging_steps               = LOGGING_STEPS,
    save_strategy               = "no",
    report_to                   = ["comet_ml"],
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = TEXT_COLUMN,
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = min(4, os.cpu_count() - 1),
    packing            = True,
    args               = training_args,
)

print("✅ Trainer configured — float32, no mixed precision")

## 🚀 Cell 8 — Train!

> GPU Memory fills in 4 buckets during training:
> - **Buckets 1–3 (fixed):** Weights + Gradients + Optimizer States → depend on model size only  
> - **Bucket 4 (variable):** Activations → grows with `batch_size × seq_length`  
>
> If you hit OOM, reduce `BATCH_SIZE` or `MAX_SEQ_LENGTH` in Cell 4 and restart.

In [ ]:
import comet_ml

# --- Pre-training memory snapshot ---
torch.cuda.reset_peak_memory_stats()
start_mem = torch.cuda.memory_allocated() / 1024**3
gpu_props  = torch.cuda.get_device_properties(0)

print("=" * 60)
print("         🚀 STARTING TRAINING LOOP")
print("=" * 60)
print(f"GPU         : {gpu_props.name}")
print(f"Total VRAM  : {gpu_props.total_memory / 1024**3:.2f} GB")
print(f"Pre-train   : {start_mem:.2f} GB used (weights + optimizer loaded)")
print()

# --- RUN ---
trainer_stats = trainer.train()

# --- Post-training diagnostics ---
end_mem  = torch.cuda.memory_allocated() / 1024**3
peak_mem = torch.cuda.max_memory_allocated() / 1024**3
total_vram = gpu_props.total_memory / 1024**3

print()
print("=" * 60)
print("         ✅ TRAINING COMPLETE")
print("=" * 60)
print(f"Steps Completed : {trainer_stats.global_step}")
print(f"Final Loss      : {trainer_stats.training_loss:.4f}")
print()
print("--- VRAM Diagnostics ---")
print(f"Peak VRAM used  : {peak_mem:.2f} / {total_vram:.2f} GB ({100*peak_mem/total_vram:.1f}% of total)")
print(f"End VRAM used   : {end_mem:.2f} GB")

if peak_mem / total_vram > 0.9:
    print("⚠️  WARNING: You used >90% VRAM. You were close to an OOM crash.")
    print("   Consider reducing BATCH_SIZE or MAX_SEQ_LENGTH in Cell 4.")
else:
    print("✅ VRAM headroom was healthy.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


         🚀 STARTING TRAINING LOOP
GPU         : Tesla T4
Total VRAM  : 14.56 GB
Pre-train   : 2.26 GB used (weights + optimizer loaded)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 440,467,456 of 596,049,920 (73.90% trained)
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/tanveer-hussain/hf-mnykgzlicbymjexhnskcxulwyjywgxutld/de9af110c2734c749ddde3b5cd8e9e11

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


Step,Training Loss
20,0.548766
40,0.498943
60,0.502890
80,0.551623
100,0.486561
120,0.596857
140,0.484190
160,0.554287
180,0.561602
200,0.615315


Step,Training Loss
20,0.548766
40,0.498943
60,0.502890
80,0.551623
100,0.486561
120,0.596857
140,0.484190
160,0.554287
180,0.561602
200,0.615315



         ✅ TRAINING COMPLETE
Steps Completed : 2500
Final Loss      : 0.5302

--- VRAM Diagnostics ---
Peak VRAM used  : 13.09 / 14.56 GB (89.9% of total)
End VRAM used   : 5.55 GB
✅ VRAM headroom was healthy.


## 💾 Cell 9 — Save & Push to Hugging Face Hub

> Full Finetuning saves the **entire model** (~1–2 GB upload for a 0.5B model),  
> not just a small adapter file like LoRA.

In [ ]:
logger.info(f"Pushing model and tokenizer to: {HUB_MODEL_ID} ...")

# Push tokenizer first (small, fast)
tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
logger.info("✅ Tokenizer pushed.")

# Push full model weights (Unsloth handles conversion back to standard HF format)
model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
logger.info("✅ Model pushed.")

print()
print("=" * 60)
print("   🎉 ALL DONE! Your model is live on the Hub.")
print("=" * 60)
print(f"\nLoad with standard HF Transformers:")
print(f"  from transformers import AutoModelForCausalLM, AutoTokenizer")
print(f"  model = AutoModelForCausalLM.from_pretrained('{HUB_MODEL_ID}')")
print()
print(f"Load with Unsloth (faster inference):")
print(f"  from unsloth import FastLanguageModel")
print(f"  model, tokenizer = FastLanguageModel.from_pretrained('{HUB_MODEL_ID}')")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6a1ilsfu/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...xuec3bl/model.safetensors:   1%|1         | 24.0MB / 2.38GB            

Saved model to https://huggingface.co/Qwen3-0.6B-Base-CPT-Math

   🎉 ALL DONE! Your model is live on the Hub.

Load with standard HF Transformers:
  from transformers import AutoModelForCausalLM, AutoTokenizer
  model = AutoModelForCausalLM.from_pretrained('Qwen3-0.6B-Base-CPT-Math')

Load with Unsloth (faster inference):
  from unsloth import FastLanguageModel
  model, tokenizer = FastLanguageModel.from_pretrained('Qwen3-0.6B-Base-CPT-Math')


## 🧪 Cell 10 — Quick Inference Test (Optional)

> Sanity-check that the model generates sensible math text after training.

In [ ]:
import warnings, transformers, torch
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
from IPython.display import display, HTML
from unsloth import FastLanguageModel

# ── Load base model ───────────────────────────────────────────────
print("Loading original base model...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float32, load_in_4bit=False, token=HF_TOKEN,
)
FastLanguageModel.for_inference(base_model)
print("✅ Base model loaded.")

# ── Generate from both models ─────────────────────────────────────
def generate(mdl, tok, prompt_text):
    inputs = tok(prompt_text, return_tensors="pt").to("cuda")
    out = mdl.generate(**inputs, max_new_tokens=200,
                       temperature=0.7, do_sample=True,
                       repetition_penalty=1.1)
    return tok.decode(out[0], skip_special_tokens=True)[len(prompt_text):].strip()

prompt = "The derivative of x^2 is"
base_out      = generate(base_model,  base_tokenizer, prompt)
finetuned_out = generate(model,       tokenizer,      prompt)

# ── Render with MathJax ───────────────────────────────────────────
html = f"""
<script>
  // Load MathJax if not already present
  if (!window.MathJax) {{
    var script = document.createElement('script');
    script.src = 'https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js';
    document.head.appendChild(script);
  }}
</script>

<style>
  .compare-box {{ font-family: Georgia, serif; max-width: 860px; margin: 16px 0; }}
  .compare-box h3 {{ font-size: 16px; margin: 0 0 8px 0; padding: 8px 12px;
                     border-radius: 6px 6px 0 0; color: white; }}
  .compare-box .body {{ padding: 14px 16px; border: 1px solid #444;
                        border-radius: 0 0 6px 6px; background: #1e1e2e;
                        color: #cdd6f4; line-height: 1.8; font-size: 14px; }}
  .base-header {{ background: #c0392b; }}
  .ft-header   {{ background: #27ae60; }}
  .prompt-box  {{ background: #2d2d44; border-left: 4px solid #89b4fa;
                  padding: 10px 14px; border-radius: 4px; color: #89b4fa;
                  font-family: monospace; margin-bottom: 16px; font-size: 14px; }}
</style>

<div class="compare-box">
  <div class="prompt-box">🧪 Prompt: {prompt}</div>

  <div style="margin-bottom:12px">
    <h3 class="compare-box base-header">❌ Base Model (no finetuning)</h3>
    <div class="body">{base_out}</div>
  </div>

  <div>
    <h3 class="compare-box ft-header">✅ Finetuned Model (CPT on Math)</h3>
    <div class="body">{finetuned_out}</div>
  </div>
</div>

<script>
  if (window.MathJax) {{ MathJax.typesetPromise(); }}
  else {{
    // Wait for MathJax to load then typeset
    setTimeout(() => {{ if (window.MathJax) MathJax.typesetPromise(); }}, 2000);
  }}
</script>
"""
display(HTML(html))

Loading original base model...


2026-03-09 07:08:50,423 - modelscope - INFO - Target directory already exists, skipping creation.


==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

/root/.cache/modelscope/hub/models/unsloth/Qwen3-0___6B-Base does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Base model loaded.
